# BTXRD official WSSS final evaluation — Kaggle only

This notebook has one scientific path: the validation-frozen WSSS U-Net. It does not train or evaluate the fully supervised diagnostic, run ablations, sweep thresholds, or regenerate pseudo masks on test.

Official checkpoint SHA-256: `02d3af8feede3c3e650cb76d664185c59092697c1c8306ea67613b89f8407fb4`; image size: 448; validation-selected threshold: 0.85. Set `BTXRD_RUN_LOCKED_TEST=1` only after `configs/official_wsss_frozen_test.json` is committed.

In [ ]:
# Cell 0 - Kaggle bootstrap: resolve immutable source, dataset, config, and checkpoint
from pathlib import Path
import hashlib
import json
import os
import shlex
import subprocess
import sys

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
REPOSITORY_URL = 'https://github.com/itsthang333/Thesis.git'
REPOSITORY_BRANCH = 'main'
SOURCE_OVERRIDE = os.environ.get('BTXRD_SOURCE_ROOT', '').strip()
if SOURCE_OVERRIDE:
    REPO_ROOT = Path(SOURCE_OVERRIDE).resolve()
elif (Path.cwd() / 'project').is_dir():
    REPO_ROOT = Path.cwd().resolve()
else:
    REPO_ROOT = KAGGLE_WORKING / 'Thesis'
    subprocess.run(['git', 'clone', '--branch', REPOSITORY_BRANCH, '--single-branch', REPOSITORY_URL, str(REPO_ROOT)], check=True)

PROJECT_DIR = REPO_ROOT / 'project'
SPLIT_MANIFEST = REPO_ROOT / 'artifacts' / 'data_audit' / 'split_manifest.csv'
FROZEN_CONFIG = REPO_ROOT / 'configs' / 'official_wsss_frozen_test.json'
BTXRD_ROOT = Path(os.environ.get('BTXRD_ROOT', '/kaggle/input/datasets/itsthang333/btxrd-raw/BTXRD'))
EXPECTED_SPLIT_SHA256 = '85511ee1bd1339c7b6b4f527acc504869da935997fd6b2485042edd619193c8c'
EXPECTED_CHECKPOINT_SHA256 = '02d3af8feede3c3e650cb76d664185c59092697c1c8306ea67613b89f8407fb4'

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not PROJECT_DIR.is_dir() or not FROZEN_CONFIG.is_file():
    raise FileNotFoundError('Attach the committed official source bundle, including the frozen config.')
if sha256_file(SPLIT_MANIFEST) != EXPECTED_SPLIT_SHA256:
    raise RuntimeError('Split-manifest hash mismatch')
if not (BTXRD_ROOT / 'images').is_dir() or not (BTXRD_ROOT / 'Annotations').is_dir():
    raise FileNotFoundError(BTXRD_ROOT)

checkpoint_override = os.environ.get('BTXRD_WSSS_CHECKPOINT', '').strip()
checkpoint_candidates = [Path(checkpoint_override)] if checkpoint_override else list(KAGGLE_INPUT.rglob('best_unet.pt'))
OFFICIAL_CHECKPOINT = next((path for path in checkpoint_candidates if path.is_file() and sha256_file(path) == EXPECTED_CHECKPOINT_SHA256), None)
if OFFICIAL_CHECKPOINT is None:
    raise FileNotFoundError('Attach best_unet.pt matching the official WSSS SHA-256')
os.environ['BTXRD_GIT_COMMIT'] = os.environ.get('BTXRD_GIT_COMMIT', '')
print({'source': str(REPO_ROOT), 'dataset': str(BTXRD_ROOT), 'checkpoint': str(OFFICIAL_CHECKPOINT), 'frozen_config': str(FROZEN_CONFIG)})


In [ ]:
# Cell 1 - verification before any test-split access
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'project', 'tests'], cwd=REPO_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_ROOT, check=True)
subprocess.run([sys.executable, 'project/tools/freeze_pipeline_config.py', '--output', str(FROZEN_CONFIG), '--verify'], cwd=REPO_ROOT, check=True)
print('Static checks, unit tests, and frozen-config verification passed.')


In [ ]:
# Cell 2 - synthetic GPU smoke test; no dataset partition is opened
import torch
sys.path.insert(0, str(PROJECT_DIR))
from models.unet import architecture_name_from_metadata, build_segmentation_model
checkpoint = torch.load(OFFICIAL_CHECKPOINT, map_location='cpu', weights_only=False)
model = build_segmentation_model(architecture_name_from_metadata(checkpoint.get('architecture')), pretrained=False)
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('The official smoke test must run on a Kaggle GPU.')
model.to(device).eval()
with torch.no_grad():
    output = model(torch.zeros(1, 3, 448, 448, device=device))
if tuple(output.shape) != (1, 1, 448, 448) or not torch.isfinite(output).all():
    raise RuntimeError(f'Invalid smoke output: {tuple(output.shape)}')
print('Synthetic GPU smoke test passed:', tuple(output.shape))


In [ ]:
# Cell 3 - the single permitted final test evaluation
RUN_FINAL_TEST = os.environ.get('BTXRD_RUN_LOCKED_TEST', '0') == '1'
OUTPUT_ROOT = Path(os.environ.get('BTXRD_TEST_OUTPUT', '/kaggle/working/official_wsss_final_test'))
if RUN_FINAL_TEST:
    if OUTPUT_ROOT.exists():
        raise FileExistsError(f'Refusing to reuse final-test output: {OUTPUT_ROOT}')
    command = [
        sys.executable, 'project/evaluate_unet.py',
        '--data-root', str(BTXRD_ROOT),
        '--split', 'test',
        '--split-manifest', str(SPLIT_MANIFEST),
        '--checkpoint', str(OFFICIAL_CHECKPOINT),
        '--frozen-config', str(FROZEN_CONFIG),
        '--image-size', '448', '--threshold', '0.85',
        '--batch-size', '8', '--num-workers', '4',
        '--bootstrap-iterations', '10000', '--bootstrap-seed', '42',
        '--prediction-dir', str(OUTPUT_ROOT / 'prediction_masks'),
        '--qualitative-dir', str(OUTPUT_ROOT / 'qualitative'),
        '--output-csv', str(OUTPUT_ROOT / 'evaluation' / 'per_image.csv'),
        '--output-json', str(OUTPUT_ROOT / 'evaluation' / 'summary.json'),
    ]
    print('$ ' + ' '.join(shlex.quote(part) for part in command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)
    summary = json.loads((OUTPUT_ROOT / 'evaluation' / 'summary.json').read_text())
    masks = list((OUTPUT_ROOT / 'prediction_masks').glob('*.png'))
    if summary.get('test_evaluated') is not True or summary.get('split') != 'test' or summary.get('images') != 373 or len(masks) != 373:
        raise RuntimeError({'summary': summary, 'prediction_masks': len(masks)})
    print(json.dumps(summary, indent=2))
else:
    print('BTXRD_RUN_LOCKED_TEST=0: test remains untouched.')
